In [1]:
# ========================
# Week 6 - Day 2: Threat Hunting & Malware Analysis
# ========================

In [2]:
import subprocess
import re
from datetime import datetime

def hunt_persistence():
    print(f"\n{'='*55}")
    print(f"🔍 HUNTING: Persistence Mechanisms")
    print('='*55)
    
    # Check systemd services (common persistence location)
    result = subprocess.run(
        ['systemctl', 'list-units', '--type=service', '--state=running'],
        capture_output=True, text=True
    )
    
    # Known legitimate services
    legitimate = [
        'sshd', 'NetworkManager', 'bluetooth', 'cups',
        'docker', 'tor', 'systemd', 'dbus', 'ufw',
        'avahi', 'nftables', 'accounts', 'polkit'
    ]
    
    suspicious = []
    lines = result.stdout.split('\n')
    
    print(f"Running services:")
    for line in lines:
        if '.service' in line:
            service_match = re.search(r'(\S+\.service)', line)
            if service_match:
                service = service_match.group(1)
                is_legit = any(leg in service.lower() for leg in legitimate)
                if is_legit:
                    print(f"  ✅ {service}")
                else:
                    print(f"  ⚠️  {service} ← UNKNOWN — investigate!")
                    suspicious.append(service)
    
    print(f"\nSuspicious services found: {len(suspicious)}")
    return suspicious

suspicious_services = hunt_persistence()


🔍 HUNTING: Persistence Mechanisms
Running services:
  ⚠️  asus-shutdown.service ← UNKNOWN — investigate!
  ⚠️  asusd.service ← UNKNOWN — investigate!
  ✅ bluetooth.service
  ⚠️  bolt.service ← UNKNOWN — investigate!
  ⚠️  containerd.service ← UNKNOWN — investigate!
  ✅ dbus-broker.service
  ✅ docker.service
  ⚠️  NetworkManager.service ← UNKNOWN — investigate!
  ⚠️  nvidia-powerd.service ← UNKNOWN — investigate!
  ⚠️  ollama.service ← UNKNOWN — investigate!
  ✅ polkit.service
  ⚠️  power-profiles-daemon.service ← UNKNOWN — investigate!
  ⚠️  rtkit-daemon.service ← UNKNOWN — investigate!
  ⚠️  sddm.service ← UNKNOWN — investigate!
  ⚠️  supergfxd.service ← UNKNOWN — investigate!
  ✅ systemd-journald.service
  ✅ systemd-logind.service
  ✅ systemd-networkd.service
  ✅ systemd-resolved.service
  ✅ systemd-timesyncd.service
  ✅ systemd-udevd.service
  ✅ systemd-userdbd.service
  ⚠️  udisks2.service ← UNKNOWN — investigate!
  ⚠️  upower.service ← UNKNOWN — investigate!
  ⚠️  user@1000.servi

In [3]:
def hunt_network_connections():
    print(f"\n{'='*55}")
    print(f"🔍 HUNTING: Suspicious Network Connections")
    print('='*55)
    
    # Get all active connections
    result = subprocess.run(
        ['ss', '-tunp'],
        capture_output=True, text=True
    )
    
    # Known legitimate ports
    legitimate_ports = {
        '22': 'SSH', '53': 'DNS', '80': 'HTTP',
        '443': 'HTTPS', '8888': 'Jupyter', '9050': 'Tor',
        '9051': 'Tor Control', '11434': 'Ollama',
        '631': 'CUPS Printing'
    }
    
    print(f"Active network connections:\n")
    lines = result.stdout.split('\n')
    
    suspicious = []
    for line in lines[1:]:  # Skip header
        if not line.strip():
            continue
            
        parts = line.split()
        if len(parts) >= 5:
            local = parts[4]
            port = local.split(':')[-1]
            
            if port in legitimate_ports:
                print(f"  ✅ Port {port:6s} ({legitimate_ports[port]:10s}) → {local}")
            else:
                print(f"  ⚠️  Port {port:6s} (Unknown) → {local}")
                suspicious.append(local)
    
    print(f"\nUnknown connections: {len(suspicious)}")
    return suspicious

hunt_network_connections()


🔍 HUNTING: Suspicious Network Connections
Active network connections:

  ⚠️  Port 68     (Unknown) → 10.61.62.186%wlan0:68
  ⚠️  Port 58875  (Unknown) → 127.0.0.1:58875
  ⚠️  Port 58875  (Unknown) → 127.0.0.1:58875
  ✅ Port 8888   (Jupyter   ) → 127.0.0.1:8888
  ⚠️  Port 41701  (Unknown) → 127.0.0.1:41701
  ⚠️  Port 47224  (Unknown) → 127.0.0.1:47224
  ⚠️  Port 49742  (Unknown) → 127.0.0.1:49742
  ⚠️  Port 50822  (Unknown) → 127.0.0.1:50822
  ⚠️  Port 50830  (Unknown) → 127.0.0.1:50830
  ⚠️  Port 50860  (Unknown) → 127.0.0.1:50860
  ⚠️  Port 51760  (Unknown) → 127.0.0.1:51760
  ⚠️  Port 51776  (Unknown) → 127.0.0.1:51776
  ⚠️  Port 52196  (Unknown) → 127.0.0.1:52196
  ⚠️  Port 52210  (Unknown) → 127.0.0.1:52210
  ⚠️  Port 52386  (Unknown) → 127.0.0.1:52386
  ✅ Port 8888   (Jupyter   ) → 127.0.0.1:8888
  ⚠️  Port 41701  (Unknown) → 127.0.0.1:41701
  ✅ Port 8888   (Jupyter   ) → 127.0.0.1:8888
  ⚠️  Port 51450  (Unknown) → 127.0.0.1:51450
  ⚠️  Port 55918  (Unknown) → 127.0.0.1:55918
  

['10.61.62.186%wlan0:68',
 '127.0.0.1:58875',
 '127.0.0.1:58875',
 '127.0.0.1:41701',
 '127.0.0.1:47224',
 '127.0.0.1:49742',
 '127.0.0.1:50822',
 '127.0.0.1:50830',
 '127.0.0.1:50860',
 '127.0.0.1:51760',
 '127.0.0.1:51776',
 '127.0.0.1:52196',
 '127.0.0.1:52210',
 '127.0.0.1:52386',
 '127.0.0.1:41701',
 '127.0.0.1:51450',
 '127.0.0.1:55918',
 '127.0.0.1:55954',
 '127.0.0.1:58875',
 '127.0.0.1:57751',
 '127.0.0.1:60913',
 '127.0.0.1:57751',
 '127.0.0.1:60913',
 '127.0.0.1:60913',
 '127.0.0.1:36516',
 '127.0.0.1:52556',
 '127.0.0.1:50094',
 '127.0.0.1:50126',
 '127.0.0.1:52068',
 '127.0.0.1:41701',
 '[2409:40f4:4433:2ba0:7441:3c6f:b92e:adcb]:40004',
 '[2409:40f4:4433:2ba0:7441:3c6f:b92e:adcb]:44764']

In [4]:
def hunt_external_connections():
    print(f"\n{'='*55}")
    print(f"🔍 HUNTING: External Connections Only")
    print('='*55)
    
    result = subprocess.run(
        ['ss', '-tunp', 'state', 'established'],
        capture_output=True, text=True
    )
    
    print(result.stdout)
    
    # Also check with netstat style
    result2 = subprocess.run(
        ['ss', '-tnp'],
        capture_output=True, text=True
    )
    
    external = []
    for line in result2.stdout.split('\n'):
        if 'ESTAB' in line:
            parts = line.split()
            if len(parts) >= 5:
                peer = parts[4]
                # Filter out localhost
                if not peer.startswith('127.') and \
                   not peer.startswith('[::1]') and \
                   not peer.startswith('[::ffff:127'):
                    print(f"🌐 External: {peer}")
                    external.append(peer)
    
    if not external:
        print("✅ No suspicious external connections found!")
    else:
        print(f"\n⚠️  {len(external)} external connections — investigate!")
    
    return external

hunt_external_connections()


🔍 HUNTING: External Connections Only
Netid Recv-Q Send-Q                             Local Address:Port          Peer Address:Port Process
udp   0      0                             10.61.62.186%wlan0:68            10.61.62.101:67   
tcp   0      0                                      127.0.0.1:58875            127.0.0.1:55918
tcp   0      0                                      127.0.0.1:58875            127.0.0.1:55954
tcp   0      0                                      127.0.0.1:8888             127.0.0.1:49742
tcp   0      0                                      127.0.0.1:41701            127.0.0.1:52196
tcp   0      0                                      127.0.0.1:47224            127.0.0.1:8888  users:(("firefox",pid=1587,fd=67))
tcp   0      0                                      127.0.0.1:49742            127.0.0.1:8888  users:(("firefox",pid=1587,fd=138))
tcp   0      0                                      127.0.0.1:50822            127.0.0.1:8888  users:(("firefox",pid=1587,fd

['[64:ff9b::226b:f35d]:443', '[2607:6bc0::10]:443']

In [5]:
import requests

external_ips = [
    "2a03:2880:f314:120:face:b00c:0:167",
    "64:ff9b::8c52:711a",
    "2404:6800:4013:813::54",
    "64:ff9b::226b:f35d",
    "2607:6bc0::10"
]

print(f"{'='*55}")
print(f"🔍 IDENTIFYING EXTERNAL CONNECTIONS")
print(f"{'='*55}\n")

for ip in external_ips:
    try:
        # Convert IPv6 to IPv4 for lookup where possible
        # 64:ff9b:: prefix means IPv4-mapped IPv6
        if ip.startswith("64:ff9b::"):
            # Extract IPv4 portion
            hex_part = ip.replace("64:ff9b::", "")
            parts = hex_part.split(":")
            if len(parts) == 2:
                oct1 = int(parts[0][:2], 16)
                oct2 = int(parts[0][2:], 16)
                oct3 = int(parts[1][:2], 16)
                oct4 = int(parts[1][2:], 16)
                ipv4 = f"{oct1}.{oct2}.{oct3}.{oct4}"
                lookup_ip = ipv4
            else:
                lookup_ip = ip
        else:
            lookup_ip = ip
            
        response = requests.get(
            f"http://ip-api.com/json/{lookup_ip}",
            timeout=5
        )
        data = response.json()
        
        print(f"IP: {ip}")
        print(f"  Owner:   {data.get('org', 'Unknown')}")
        print(f"  Country: {data.get('country', 'Unknown')}")
        print(f"  City:    {data.get('city', 'Unknown')}")
        
        # Risk assessment
        trusted = ['facebook', 'google', 'mozilla',
                  'cloudflare', 'meta', 'fastly', 'anthropic']
        org = data.get('org', '').lower()
        is_trusted = any(t in org for t in trusted)
        
        if is_trusted:
            print(f"  Status:  ✅ TRUSTED — {data.get('org')}")
        else:
            print(f"  Status:  ⚠️  UNKNOWN — investigate further!")
        print()
        
    except Exception as e:
        print(f"IP: {ip} → Error: {e}\n")

🔍 IDENTIFYING EXTERNAL CONNECTIONS

IP: 2a03:2880:f314:120:face:b00c:0:167
  Owner:   Facebook, Inc
  Country: India
  City:    Mastemau
  Status:  ✅ TRUSTED — Facebook, Inc

IP: 64:ff9b::8c52:711a
  Owner:   GitHub, Inc.
  Country: United States
  City:    San Francisco
  Status:  ⚠️  UNKNOWN — investigate further!

IP: 2404:6800:4013:813::54
  Owner:   Google Public DNS (del)
  Country: India
  City:    Delhi
  Status:  ✅ TRUSTED — Google Public DNS (del)

IP: 64:ff9b::226b:f35d
  Owner:   Google Cloud
  Country: United States
  City:    Kansas City
  Status:  ✅ TRUSTED — Google Cloud

IP: 2607:6bc0::10
  Owner:   Anthropic, PBC
  Country: United States
  City:    San Francisco
  Status:  ✅ TRUSTED — Anthropic, PBC

